# Chapter 7, Guide 3: Linear combination of unitaries

Companion notebook to *Quantum Algorithms from First Principles*, Chapter 7,
"Building an LCU".

$A=\sum_i c_iU_i$ with $c_i>0$. Prepare $O|0\rangle=\|c\|_1^{-1/2}\sum_i\sqrt{c_i}|i\rangle$,
select $V=\sum_i|i\rangle\langle i|\otimes U_i$, apply, in circuit order,
$O$, then $V$, then $O^\dagger$, and post-select the index register on $|0\rangle$. The
book's tests: $O$ unitary with the right amplitudes; $V$ equal to its definition; the
ancilla-$|0\rangle$ block equal to $A|\psi\rangle/\|c\|_1$; $p=\|A|\psi\rangle\|^2/\|c\|_1^2$.

Three constructions of $O$ are shown: Qiskit's generic state preparation, the
controlled-$R_y$ cascade for a handful of terms, and the binary-tree preparation the
book recommends beyond that. Negative coefficients are handled by absorbing the sign
into $U_i$. Convention: book wire $j$ (top, most significant) is Qiskit qubit `bq(j)`.

In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector, SparsePauliOp
from qiskit.circuit.library import StatePreparation

bq = lambda j, tot: tot - 1 - j

## The book's worked case: $A = 0.6X + 1.4Z$, $p = 0.580$ for every $|\psi\rangle$

In [2]:
coeffs = [0.6, 1.4]
paulis = ["X", "Z"]
norm1 = sum(coeffs)
A = sum(c * SparsePauliOp(P).to_matrix() for c, P in zip(coeffs, paulis))

## Prepare $O$ by a binary tree of controlled $R_y$ rotations

One $R_y$ per internal node: at depth $d$ the node for prefix $b$ rotates the next qubit
by $2\arccos\sqrt{w_{b0}/w_b}$ where $w_b$ is the total weight under that prefix,
controlled on the prefix. $m-1$ controlled rotations for $m$ leaves.

In [3]:
def prepare_tree(weights, qubits):
    """Circuit on len(qubits) qubits (book order, qubits[0] most significant) sending |0>
    to sum_i sqrt(w_i/sum w) |i>. Pads weights to a power of two with zeros."""
    k = len(qubits)
    w = np.zeros(2**k); w[:len(weights)] = weights
    qc = QuantumCircuit(max(qubits) + 1)
    for depth in range(k):
        block = 2**(k - depth)                       # leaves under one node at this depth
        for node in range(2**depth):
            lo = node * block
            tot_w = w[lo:lo + block].sum()
            if tot_w == 0:
                continue
            left = w[lo:lo + block // 2].sum()
            angle = 2 * np.arccos(np.sqrt(left / tot_w))
            if depth == 0:
                qc.ry(angle, qubits[0])
            else:
                prefix = [(node >> (depth - 1 - i)) & 1 for i in range(depth)]
                ctrls = qubits[:depth]
                zeros = [q for q, bit in zip(ctrls, prefix) if bit == 0]
                if zeros: qc.x(zeros)
                qc.mcry(angle, ctrls, qubits[depth])
                if zeros: qc.x(zeros)
    return qc


def prepare_qiskit(weights, qubits):
    k = len(qubits)
    amp = np.zeros(2**k); amp[:len(weights)] = np.sqrt(np.array(weights) / sum(weights))
    qc = QuantumCircuit(max(qubits) + 1)
    qc.append(StatePreparation(amp), qubits[::-1])    # Qiskit wants LSB first
    return qc


# Test: O|0> has amplitudes sqrt(c_i / ||c||_1), and O is unitary
k = 1
O = prepare_tree(coeffs, [0])
amps = Statevector(O).data
print("O|0> amplitudes:", np.round(amps.real, 5), " expected:", np.round(np.sqrt(np.array(coeffs) / norm1), 5))
assert np.allclose(amps, np.sqrt(np.array(coeffs) / norm1))
assert np.allclose(Operator(O).data @ Operator(O).data.conj().T, np.eye(2))
print("O test passed")

O|0> amplitudes: [0.54772 0.83666]  expected: [0.54772 0.83666]
O test passed


## Select $V$ and the full $U_{\rm LCU}$

In [4]:
def select(qc, idx_qubits, sys_qubits, unitaries):
    """V = sum_i |i><i| (x) U_i, with each U_i given as a (pauli string, phase) pair."""
    k = len(idx_qubits)
    for i, (P, phase) in enumerate(unitaries):
        bits = [(i >> (k - 1 - d)) & 1 for d in range(k)]
        zeros = [q for q, bit in zip(idx_qubits, bits) if bit == 0]
        if zeros: qc.x(zeros)
        gate = SparsePauliOp(P, coeffs=[phase]).to_operator().to_instruction().control(k)
        qc.append(gate, idx_qubits + sys_qubits[::-1])      # Operator -> LSB-first system order
        if zeros: qc.x(zeros)


def lcu_circuit(coeffs, unitaries, n_sys, prep=prepare_tree):
    m = len(coeffs); k = max(1, int(np.ceil(np.log2(m))))
    tot = k + n_sys
    idx = [bq(j, tot) for j in range(k)]
    sysq = [bq(k + j, tot) for j in range(n_sys)]
    qc = QuantumCircuit(tot)
    O = prep(coeffs, list(range(k)))                   # built on k qubits in book order 0..k-1
    omap = {q: idx[q] for q in range(k)}
    qc.compose(O, qubits=[omap[q] for q in range(k)], inplace=True)
    select(qc, idx, sysq, unitaries)
    qc.compose(O.inverse(), qubits=[omap[q] for q in range(k)], inplace=True)
    return qc, idx, sysq


def zero_block(qc, idx, sysq):
    """The system operator obtained by projecting the index register onto |0> in and out."""
    U = Operator(qc).data
    tot = qc.num_qubits; n_sys = len(sysq)
    # index register occupies the top k book wires = the most significant bits of the row index
    rows = np.arange(2**n_sys)                         # index register = 0 -> row index < 2^n_sys
    return U[np.ix_(rows, rows)]


unitaries = [("X", 1.0), ("Z", 1.0)]
qc, idx, sysq = lcu_circuit(coeffs, unitaries, 1)
blk = zero_block(qc, idx, sysq)
print("ancilla-|0> block * ||c||_1:\n", np.round(blk * norm1, 6).real)
assert np.allclose(blk, A / norm1)
print("Block test passed: block = A/||c||_1")

# Test V alone against its definition
qcV = QuantumCircuit(2); select(qcV, [1], [0], unitaries)
V_def = np.kron(np.diag([1, 0]), SparsePauliOp("X").to_matrix()) + np.kron(np.diag([0, 1]), SparsePauliOp("Z").to_matrix())
assert np.allclose(Operator(qcV).data, V_def)
print("V test passed")

ancilla-|0> block * ||c||_1:
 [[ 1.4  0.6]
 [ 0.6 -1.4]]
Block test passed: block = A/||c||_1
V test passed


## Success probability for several $|\psi\rangle$: $p = \|A|\psi\rangle\|^2/\|c\|_1^2$

In [5]:
rng = np.random.default_rng(0)
for trial in range(3):
    psi = rng.normal(size=2) + 1j * rng.normal(size=2); psi /= np.linalg.norm(psi)
    full = QuantumCircuit(2)
    full.initialize(psi, [0])                       # system is Qiskit qubit 0 here
    full.compose(qc, inplace=True)
    sv = Statevector(full)
    p_circ = sv.probabilities([1])[0]               # index register (Qiskit qubit 1) == 0
    p_formula = np.linalg.norm(A @ psi) ** 2 / norm1**2
    print(f"trial {trial}: p_circuit = {p_circ:.4f}, formula = {p_formula:.4f}")
    assert abs(p_circ - p_formula) < 1e-10
print("p = 0.580 for every state, since A^2 = (c0^2 + c1^2) I here")

trial 0: p_circuit = 0.5800, formula = 0.5800
trial 1: p_circuit = 0.5800, formula = 0.5800
trial 2: p_circuit = 0.5800, formula = 0.5800
p = 0.580 for every state, since A^2 = (c0^2 + c1^2) I here


## More terms, negative coefficients, and the three preparations

$A = 0.5\,XI - 0.3\,ZZ + 0.2\,IY + 0.4\,XX$ on two qubits. The negative coefficient is
absorbed as $U_1' = -ZZ$ (phase $e^{i\pi}$), leaving all $c_i$ positive.

In [6]:
terms = [("XI", 0.5), ("ZZ", -0.3), ("IY", 0.2), ("XX", 0.4)]
coeffs2 = [abs(c) for _, c in terms]
unitaries2 = [(P, np.sign(c)) for P, c in terms]
A2 = sum(c * SparsePauliOp(P).to_matrix() for P, c in terms)
norm2 = sum(coeffs2)
for name, prep in [("binary tree", prepare_tree), ("Qiskit StatePreparation", prepare_qiskit)]:
    qc2, idx2, sys2 = lcu_circuit(coeffs2, unitaries2, 2, prep=prep)
    blk2 = zero_block(qc2, idx2, sys2)
    assert np.allclose(blk2, A2 / norm2), name
    print(f"{name}: block = A/||c||_1 with 4 terms and a negative coefficient: OK")

binary tree: block = A/||c||_1 with 4 terms and a negative coefficient: OK
Qiskit StatePreparation: block = A/||c||_1 with 4 terms and a negative coefficient: OK


## Ancilla count for the Taylor-series version

For $e^{-iHt}\approx\sum_{k\le K}\frac{(-it)^k}{k!}H^k$ with $H=\sum_{j<m}c_jP_j$, the index
must label a product of up to $K$ of the $P_j$: $K\lceil\log_2 m\rceil$ qubits, plus
$\lceil\log_2(K+1)\rceil$ to hold the order $k$.

In [7]:
for m, K in [(4, 3), (15, 6), (100, 10)]:
    print(f"m={m:3d}, K={K:2d}: index ancillas = {K*int(np.ceil(np.log2(m)))}, order register = {int(np.ceil(np.log2(K+1)))}")

m=  4, K= 3: index ancillas = 6, order register = 2
m= 15, K= 6: index ancillas = 24, order register = 3
m=100, K=10: index ancillas = 70, order register = 4
